# 00 · Data acquisition and description

**Project:** *Detecting Learnable Signal in Tabular Datasets Before Model Training: A Comparative Evaluation of Statistical, Geometric and Model-Based Probes*

This notebook downloads, verifies and documents every real dataset used in the study and writes clean copies to `../data/processed/`. All later notebooks read only from that folder.

| Role | Dataset | Rows | Why it is used |
|---|---|---|---|
| **Signal (knowledge)** | Adult / Census Income (UCI) | 48,842 | Required by the supervisor; income >50K is known to be predictable from demographic and work attributes |
| **No signal by design** | NSW job-training experiment, Dehejia–Wahba sample | 445 | Target = *randomised* treatment assignment, so it is independent of pre-treatment covariates by construction |
| **No signal by design** (second case) | HIV-results incentive experiment, Malawi (Thornton data) | 2,901 | Target = whether a person was *randomly* offered a cash incentive; individual covariates measured before the offer (village excluded, see Section 4) |
| Additional signal datasets (semi-synthetic, Ojala–Garriga replication) | Pima Indians Diabetes, Banknote Authentication, Breast Cancer Wisconsin (Diagnostic) | 768 / 1,372 / 569 | Small benchmark datasets with known, differing difficulty |

**Where the files come from.** The build environment for this study could reach GitHub but not UCI, OpenML or PyPI. Copies of the datasets are therefore taken from two public GitHub repositories:

* `jbrownlee/Datasets` – mirrors of the UCI files (Adult, Pima, Banknote).
* `NickCH-K/causaldata` – the `nsw_mixtape` data used in Cunningham's *Causal Inference: The Mixtape*, i.e. the Dehejia–Wahba experimental subsample of the NSW programme, and the `thornton_hiv` data from Thornton (2008).

Breast Cancer Wisconsin ships with scikit-learn. If you run this notebook elsewhere, the original UCI files (`https://archive.ics.uci.edu`) can be used instead; the loader functions below accept a local path.

In [1]:
import os
import subprocess
import hashlib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

RAW = os.path.abspath("../data/raw")
PROC = os.path.abspath("../data/processed")
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)
print("raw folder:", RAW)
print("processed folder:", PROC)

raw folder: /home/claude/research/data/raw
processed folder: /home/claude/research/data/processed


## 1. Download helpers

`git_sparse_fetch` clones only the files we need from a GitHub repository (sparse checkout), so the download stays small. Files already present in `data/raw` are not downloaded again.

In [2]:
def git_sparse_fetch(repo_url, files, dest_dir):
    """Download selected files from a GitHub repository via a shallow sparse clone.

    Parameters
    ----------
    repo_url : str   – HTTPS URL of the repository.
    files    : list  – paths inside the repository to fetch.
    dest_dir : str   – folder into which the files are copied (flat).

    Returns the list of local file paths. Existing files are reused.
    """
    targets = [os.path.join(dest_dir, os.path.basename(f)) for f in files]
    if all(os.path.exists(t) for t in targets):
        return targets
    tmp = os.path.join(dest_dir, "_clone_" + hashlib.md5(repo_url.encode()).hexdigest()[:8])
    if not os.path.exists(tmp):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, tmp],
                       check=True, capture_output=True)
    subprocess.run(["git", "-C", tmp, "sparse-checkout", "set", "--no-cone"] + ["/" + f for f in files],
                   check=True, capture_output=True)
    for f, t in zip(files, targets):
        with open(os.path.join(tmp, f), "rb") as src, open(t, "wb") as dst:
            dst.write(src.read())
    return targets


def sha256(path):
    """SHA-256 checksum of a file, recorded for reproducibility."""
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

In [3]:
uci_files = git_sparse_fetch("https://github.com/jbrownlee/Datasets.git",
                             ["adult-all.csv", "pima-indians-diabetes.csv", "banknote_authentication.csv"], RAW)
nsw_files = git_sparse_fetch("https://github.com/NickCH-K/causaldata.git",
                             ["Python/causaldata/nsw_mixtape/nsw_mixtape.dta",
                              "Python/causaldata/thornton_hiv/thornton_hiv.dta"], RAW)
for f in uci_files + nsw_files:
    print(f"{os.path.basename(f):32s} {os.path.getsize(f)/1e6:7.3f} MB  sha256={sha256(f)[:16]}…")

adult-all.csv                      5.277 MB  sha256=21e0ea2f925a0033…
pima-indians-diabetes.csv          0.023 MB  sha256=6bfe5d0f379d17a0…
banknote_authentication.csv        0.046 MB  sha256=d0539aaed2139ba7…
nsw_mixtape.dta                    0.025 MB  sha256=e4a64e4436c2c178…
thornton_hiv.dta                   0.076 MB  sha256=add8c0410f44f056…


## 2. Adult (signal dataset)

The Adult file has no header row. Column names follow the UCI documentation. Missing values are coded as `?`. The target is `income > 50K`. The column `fnlwgt` is a census sampling weight, not a personal attribute; it is kept, as in most benchmark uses, but noted. `education` and `education-num` encode the same information; both are kept because a sanity check should not silently edit the data.

In [4]:
ADULT_COLS = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status", "occupation",
              "relationship", "race", "sex", "capital_gain", "capital_loss", "hours_per_week",
              "native_country", "income"]
adult = pd.read_csv(os.path.join(RAW, "adult-all.csv"), header=None, names=ADULT_COLS,
                    na_values="?", skipinitialspace=True)
adult["y"] = adult["income"].str.strip().str.replace(".", "", regex=False).eq(">50K").astype(int)
adult = adult.drop(columns="income")
print("shape:", adult.shape)
print("positive rate (income > 50K):", round(adult.y.mean(), 4))
print("missing values per column:\n", adult.isna().sum()[adult.isna().sum() > 0])
adult.head()

shape: (48842, 15)
positive rate (income > 50K): 0.2393
missing values per column:
 workclass         2799
occupation        2809
native_country     857
dtype: int64


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,y
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


## 3. NSW randomised experiment (no-signal dataset)

The National Supported Work (NSW) Demonstration randomly assigned applicants to a job-training programme (`treat = 1`) or to a control group (`treat = 0`). We use the Dehejia–Wahba experimental subsample (445 people: 185 treated, 260 controls).

**Why it has no learnable signal by design.** The target is the *treatment assignment*, decided by a lottery. Pre-treatment characteristics (age, education, ethnicity, marital status, degree, earnings in 1974 and 1975) therefore carry no information about it in the population. Any detected "signal" is a false positive (up to chance imbalance, which H₀ permits).

**Important:** the column `re78` (earnings in 1978) was measured *after* treatment and is affected by it. Including it would introduce genuine but post-outcome information, which is a textbook example of leakage. We therefore build two versions:

* `nsw_null` – pre-treatment covariates only (the no-signal dataset);
* `nsw_leak` – the same plus `re78` (used in the leakage experiment, notebook 06).

In [5]:
nsw = pd.read_stata(os.path.join(RAW, "nsw_mixtape.dta"))
nsw = nsw.drop(columns=["data_id"]).rename(columns={"treat": "y"})
nsw["y"] = nsw["y"].astype(int)
pre_treatment = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]
nsw_null = nsw[pre_treatment + ["y"]].astype(float).astype({"y": int})
nsw_leak = nsw[pre_treatment + ["re78", "y"]].astype(float).astype({"y": int})
print("NSW shape:", nsw.shape, "| treated:", nsw.y.sum(), "controls:", (1 - nsw.y).sum())
nsw_null.describe().T.round(2)

NSW shape: (445, 10) | treated: 185 controls: 260


,count,mean,std,min,25%,50%,75%,max
age,445.0,25.37,7.10,17.0,20.0,24.0,28.00,55.00
educ,445.0,10.20,1.79,3.0,9.0,10.0,11.00,16.00
black,445.0,0.83,0.37,0.0,1.0,1.0,1.00,1.00
hisp,445.0,0.09,0.28,0.0,0.0,0.0,0.00,1.00
marr,445.0,0.17,0.37,0.0,0.0,0.0,0.00,1.00
nodegree,445.0,0.78,0.41,0.0,1.0,1.0,1.00,1.00
re74,445.0,2102.27,5363.58,0.0,0.0,0.0,824.39,39570.68
re75,445.0,1377.14,3150.96,0.0,0.0,0.0,1220.84,25142.24
y,445.0,0.42,0.49,0.0,0.0,0.0,1.00,1.00


A classical balance check (standardised mean differences between treated and controls) confirms that randomisation produced comparable groups. Values below about 0.1–0.2 are conventionally read as balanced.

In [6]:
def standardised_mean_difference(df, cols, y="y"):
    """SMD = (mean_treated - mean_control) / pooled SD, per covariate."""
    t, c = df[df[y] == 1], df[df[y] == 0]
    sd = np.sqrt((t[cols].var() + c[cols].var()) / 2)
    return ((t[cols].mean() - c[cols].mean()) / sd).rename("SMD")

standardised_mean_difference(nsw_null, pre_treatment).round(3).to_frame()

,SMD
age,0.107
educ,0.141
black,0.044
hisp,-0.175
marr,0.094
nodegree,-0.304
re74,-0.002
re75,0.084


## 4. Thornton (2008) HIV incentive experiment (second no-signal dataset)

In rural Malawi, people tested for HIV were randomly offered cash vouchers (drawn by lottery) for collecting their results at a nearby centre. The target `any` = 1 if the person received any incentive. It was assigned at random, so it is independent of the pre-treatment covariates:

* `age`, `hiv2004` (the HIV test result from the baseline test) and `distvct` (distance to the results centre, whose location was itself randomised).

**Why village is excluded from the null dataset.** A first analysis (notebook 02) found that classifiers predict incentive receipt far better than chance once the village is included. The check below shows why: the share of people receiving an incentive differs strongly between villages, i.e. assignment probabilities were not constant across villages. With village included, the target genuinely depends on a feature, so that version (`thornton_village`) is kept as an instructive extra case, not as a null.

Excluded columns: `got` (whether results were collected) is the experiment's **outcome**, measured after treatment. `tinc` (incentive amount) **determines** the target exactly (`tinc > 0` ⇔ `any = 1`), so it is a perfect leak; it is kept only in `thornton_leak` for the leakage experiment. Rows with a missing target (people not in the incentive lottery) are dropped.

In [7]:
th = pd.read_stata(os.path.join(RAW, "thornton_hiv.dta"))
th = th[th["any"].notna()].copy()
th["y"] = th["any"].astype(int)
th["villnum"] = th["villnum"].map(lambda v: f"v{int(v)}" if pd.notna(v) else np.nan)
thornton_null = th[["age", "hiv2004", "distvct", "y"]].reset_index(drop=True)
thornton_village = th[["age", "hiv2004", "distvct", "villnum", "y"]].reset_index(drop=True)
thornton_leak = th[["age", "hiv2004", "distvct", "tinc", "y"]].reset_index(drop=True)
print("Thornton shape:", thornton_null.shape, "| incentive:", thornton_null.y.sum(),
      "no incentive:", (1 - thornton_null.y).sum(), "| villages:", thornton_village.villnum.nunique())
print("check: tinc > 0 exactly when any = 1 ->", bool(((thornton_leak.tinc > 0) == (thornton_leak.y == 1)).all()))
standardised_mean_difference(thornton_null, ["age", "hiv2004", "distvct"]).round(3).to_frame()

Thornton shape: (2901, 4) | incentive: 2222 no incentive: 679 | villages: 119
check: tinc > 0 exactly when any = 1 -> True


,SMD
age,0.094
hiv2004,0.000
distvct,0.072


Assignment rate by village: a chi-square test of independence between village and incentive receipt.

In [8]:
from scipy import stats as _st
tab = pd.crosstab(thornton_village.villnum.fillna("missing"), thornton_village.y)
chi2, p_village, dof, _ = _st.chi2_contingency(tab)
rate = thornton_village.groupby(thornton_village.villnum.fillna("missing")).y.mean()
print(f"chi2 = {chi2:.1f}, dof = {dof}, p = {p_village:.2e}")
print(f"incentive rate across villages: 10th percentile {rate.quantile(0.1):.2f}, median {rate.median():.2f}, 90th percentile {rate.quantile(0.9):.2f}")

chi2 = 402.2, dof = 119, p = 2.01e-32
incentive rate across villages: 10th percentile 0.53, median 0.80, 90th percentile 0.97


## 5. Additional benchmark datasets

In [9]:
pima = pd.read_csv(os.path.join(RAW, "pima-indians-diabetes.csv"), header=None,
                   names=["pregnancies", "glucose", "blood_pressure", "skin_thickness", "insulin",
                          "bmi", "pedigree", "age", "y"])
banknote = pd.read_csv(os.path.join(RAW, "banknote_authentication.csv"), header=None,
                       names=["variance", "skewness", "curtosis", "entropy", "y"])
bc = load_breast_cancer(as_frame=True)
breast = bc.frame.rename(columns={"target": "y"})
breast.columns = [c.replace(" ", "_") for c in breast.columns]
for name, df in [("pima", pima), ("banknote", banknote), ("breast_cancer", breast)]:
    print(f"{name:14s} shape={df.shape}  positive rate={df.y.mean():.3f}")

pima           shape=(768, 9)  positive rate=0.349
banknote       shape=(1372, 5)  positive rate=0.445
breast_cancer  shape=(569, 31)  positive rate=0.627


## 6. Save processed copies and a dataset register

In [10]:
datasets = {"adult": adult, "nsw_null": nsw_null, "nsw_leak": nsw_leak,
            "thornton_null": thornton_null, "thornton_village": thornton_village, "thornton_leak": thornton_leak,
            "pima": pima, "banknote": banknote, "breast_cancer": breast}
register = []
for name, df in datasets.items():
    path = os.path.join(PROC, f"{name}.csv")
    df.to_csv(path, index=False)
    register.append({"dataset": name, "rows": len(df), "features": df.shape[1] - 1,
                     "categorical_features": int((df.drop(columns="y").dtypes == object).sum()),
                     "positive_rate": round(df.y.mean(), 4), "file": os.path.basename(path)})
register = pd.DataFrame(register)
register.to_csv(os.path.join(PROC, "dataset_register.csv"), index=False)
register

,dataset,rows,features,categorical_features,positive_rate,file
0,adult,48842,14,0,0.2393,adult.csv
1,nsw_null,445,8,0,0.4157,nsw_null.csv
2,nsw_leak,445,9,0,0.4157,nsw_leak.csv
3,thornton_null,2901,3,0,0.7659,thornton_null.csv
4,thornton_village,2901,4,0,0.7659,thornton_village.csv
5,thornton_leak,2901,4,0,0.7659,thornton_leak.csv
6,pima,768,8,0,0.3490,pima.csv
7,banknote,1372,4,0,0.4446,banknote.csv
8,breast_cancer,569,30,0,0.6274,breast_cancer.csv
